In [17]:
import wandb
wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [18]:
pip install thop

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from thop import profile
import wandb

# --- 1. Custom Made Dataloader ---
class CustomCIFAR10(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.dataset = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 2. Training Logic ---
def run_experiment():
    wandb.init(project="cifar10-analysis", name="resnet18-30-epochs")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_set = CustomCIFAR10(root='./data', train=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)

    model = torchvision.models.resnet18(num_classes=10).to(device)

    # --- 3. Count FLOPs ---
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    wandb.config.update({"total_flops": flops, "parameters": params})
    print(f"Total FLOPs: {flops / 1e6:.2f} Million")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    # Training Loop
    for epoch in range(30): # Training for 30 epochs
        model.train()
        running_loss = 0.0

        old_weights = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

        for i, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()

            # --- 4. Visualize Gradient Flow (log every 100 steps) ---
            if i % 100 == 0:
                grad_dict = {}
                for n, p in model.named_parameters():
                    if p.grad is not None and "weight" in n:
                        grad_dict[f"grad_flow/{n}"] = p.grad.abs().mean().item()
                wandb.log(grad_dict)

            optimizer.step()
            running_loss += loss.item()

        # --- 5. Visualize Weight Update Flow ---
        update_dict = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                update_mag = (p - old_weights[n]).abs().mean().item()
                update_dict[f"weight_update/{n}"] = update_mag

        wandb.log({
            "epoch": epoch + 1,
            "loss": running_loss / len(train_loader),
            "lr": scheduler.get_last_lr()[0],
            **update_dict
        })

        print(f"Epoch {epoch+1}/30 - Loss: {running_loss/len(train_loader):.4f}")
        scheduler.step()

    wandb.finish()

run_experiment()

Using device: cuda
Total FLOPs: 37.22 Million
Epoch 1/30 - Loss: 2.1622
Epoch 2/30 - Loss: 1.5380
Epoch 3/30 - Loss: 1.3485
Epoch 4/30 - Loss: 1.1794
Epoch 5/30 - Loss: 1.0718
Epoch 6/30 - Loss: 0.9953
Epoch 7/30 - Loss: 0.9419
Epoch 8/30 - Loss: 0.8893
Epoch 9/30 - Loss: 0.8556
Epoch 10/30 - Loss: 0.8167
Epoch 11/30 - Loss: 0.7811
Epoch 12/30 - Loss: 0.7511
Epoch 13/30 - Loss: 0.7246
Epoch 14/30 - Loss: 0.6927
Epoch 15/30 - Loss: 0.6654
Epoch 16/30 - Loss: 0.6389
Epoch 17/30 - Loss: 0.6089
Epoch 18/30 - Loss: 0.5890
Epoch 19/30 - Loss: 0.5595
Epoch 20/30 - Loss: 0.5312
Epoch 21/30 - Loss: 0.5002
Epoch 22/30 - Loss: 0.4676
Epoch 23/30 - Loss: 0.4397
Epoch 24/30 - Loss: 0.4096
Epoch 25/30 - Loss: 0.3676
Epoch 26/30 - Loss: 0.3442
Epoch 27/30 - Loss: 0.3118
Epoch 28/30 - Loss: 0.2943
Epoch 29/30 - Loss: 0.2781
Epoch 30/30 - Loss: 0.2668


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
grad_flow/bn1.weight,█▁▁▁▂▂▂▂▂▂▂▃▂▂▂▃▃▂▂▂▂▂▃▄▂▂▃▃▃▃▃▃▃▃▄▃▄▃▄▂
grad_flow/conv1.weight,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grad_flow/fc.weight,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grad_flow/layer1.0.bn1.weight,█▂▁▁▁▁▁▂▂▂▃▃▂▂▂▃▄▃▃▃▃▃▃▄▄▅▄▃▄▅▅▄▅▅▄▄▅▅▅▆
grad_flow/layer1.0.bn2.weight,▇▂▁▂▁▂▃▄▃▃▃▃▅▄▄▄▄▄▄▄▄▄▅▇▄▄▄▅▄▄▆▄▅▅▆▆█▆▆▆
grad_flow/layer1.0.conv1.weight,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁
grad_flow/layer1.0.conv2.weight,▁▂▁▂▂▂▂▂▃▂▂▂▃▂▃▃▃▄▃▅▄▃▅▅▄▄▅▄▆▄▆▅█▄▆▅▄█▆▅
grad_flow/layer1.1.bn1.weight,▁▁▁▁▁▂▂▂▃▃▄▂▃▄▄▄▄▄▄▄▅▅▆▅▄▅▆▅▅▆▆▅▄▄▅▇▇▇▆█
grad_flow/layer1.1.bn2.weight,█▁▃▂▃▄▆▄▃▃▄▅▄▇▆▄▄▄▅▅▅▄▇▇▄▆▆▄▇▆▆▇▆▆▇▆█▆██
+96,...
